# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
        }
     # loop up to `tries` times
    for attempt in range(tries):
        try:
            # request with a timeout
            r = requests.get(BASE, params=params, timeout=10)
            # If successful, return the daily weather data as a df
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()

        except requests.exceptions.Timeout:
            # Retry if the request times out
            pass

        except requests.exceptions.ConnectionError:
            pass
        # If attempts remain, wait an increasing amount of time
        if attempt < tries - 1:
            time.sleep(2 ** attempt)

    # Raise if every attempt failed
    raise RuntimeError(f'Weather request failed after {tries} attempts')



    # TODO: loop up to `tries` times
    # TODO: request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO: raise on a status that is not in RETRY_STATUSES
    # TODO: otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about


used_fallback = False

d1, d2 = '2026-09-05', '2026-10-03'
WEEKENDS = pd.to_datetime([d1, d2])
span = get_weather(d1, d2)
span['time'] = pd.to_datetime(span['time'])
weekends = span[span['time'].isin(WEEKENDS)].reset_index(drop=True)

assert len(weekends) == 2, f'expected 2 weekend rows, got {len(weekends)}'

display(weekends)


,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
1,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})


used_fallback = False
# Fallback is not used in part 2 but below is the code for the TODOs.
wx = weekends.copy()
wx['source'] = 'open-meteo'

display(wx)
print('used_fallback:', used_fallback)

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


used_fallback: False


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO

wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter['precip_mm'] - drier['precip_mm']

print(
    f"October 3, 2026, was wetter than September 5, 2026, "
    f"by {difference:.1f} mm of precipitation."
)

October 3, 2026, was wetter than September 5, 2026, by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO

from datetime import datetime

# API citation metadata
endpoint = 'https://archive-api.open-meteo.com/v1/archive'
lat, lon = 38.03, -78.51
start_date, end_date = '2026-09-05', '2026-10-03'
pulled_at = datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')

# Print citation details
print('Weather Data Citation')
print('Endpoint:', endpoint)
print(f'Coordinates: latitude {lat}, longitude {lon}')
print(f'Date range: {start_date} to {end_date}')
print('Pulled at:', pulled_at)
print('Data status:', 'Fallback sample' if used_fallback else 'Live API data (Open-Meteo)')

Weather Data Citation
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-12 01:27:47 UTC
Data status: Live API data (Open-Meteo)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather() returned an empty DataFrame instead of raising an error, the merge would still run, but the weather columns would contain missing values for all 200 days of sales. The chart would show no meaningful relationship between rain and revenue because there would be no actual weather data to plot. Nobody might catch the problem because the merge  succeeds without an error, and the chart may appear blank. This is why a function should raise an error or label fallback data instead of returning an empty DataFrame.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch

d1, d2 = '2026-09-05', '2026-10-03'
charlottesville = get_weather(d1, d2, lat=38.03, lon=-78.51)
charlottesville['city'] = 'Charlottesville'
orlando = get_weather(d1, d2, lat=28.54, lon=-81.38)
orlando['city'] = 'Orlando'

weather_both = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)

weather_both = weather_both.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

weather_both['date'] = pd.to_datetime(weather_both['date'])
weather_both['source'] = 'open-meteo'
weather_both = weather_both[
    ['date', 'city', 'temp_max', 'precip_mm', 'source']
]

display(weather_both)

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
1,2026-09-06,Charlottesville,27.2,1.6,open-meteo
2,2026-09-07,Charlottesville,27.7,0.0,open-meteo
3,2026-09-08,Charlottesville,29.4,0.0,open-meteo
4,2026-09-09,Charlottesville,30.9,0.0,open-meteo
5,2026-09-10,Charlottesville,34.2,3.2,open-meteo
6,2026-09-11,Charlottesville,28.0,10.4,open-meteo
7,2026-09-12,Charlottesville,27.5,21.5,open-meteo
8,2026-09-13,Charlottesville,31.3,5.8,open-meteo
9,2026-09-14,Charlottesville,28.7,0.3,open-meteo
